In [1]:
from pathlib import Path
import os
import pandas as pd
from collections import Counter
import numpy as np

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform


In [2]:
# --------------------------------------------------
# 1. Define dataset path
# --------------------------------------------------

data_set_path = (
    Path.home()
    / "Desktop"
    / "fighting-churn-learning"
    / "output"
    / "churn_dataset.csv"
)


# Calculating the correlation matrix for a dataset in Python

In [3]:
# --------------------------------------------------
# Define Listing 6.2 function
# --------------------------------------------------

def dataset_correlation_matrix(data_set_path):

    # ① Check that dataset exists
    assert os.path.isfile(data_set_path), \
        f'"{data_set_path}" is not a valid path'

    # ② Load dataset
    # account_id and observation_date become index
    churn_data = pd.read_csv(
        data_set_path,
        index_col=[0, 1]
    )

    # Convert churn target to numeric if necessary
    if "is_churn" in churn_data.columns:

        if churn_data["is_churn"].dtype == object:

            churn_data["is_churn"] = (
                churn_data["is_churn"]
                .astype(str)
                .str.lower()
                .map({
                    "true": 1.0,
                    "false": 0.0,
                    "t": 1.0,
                    "f": 0.0,
                    "1": 1.0,
                    "0": 0.0
                })
            )

        else:
            churn_data["is_churn"] = (
                churn_data["is_churn"]
                .astype(float)
            )

    # Keep numeric columns only
    churn_data = churn_data.select_dtypes(
        include="number"
    )

    # ③ Sort columns alphabetically
    churn_data = churn_data.reindex(
        sorted(churn_data.columns),
        axis=1
    )

    # ④ Calculate Pearson correlation matrix
    corr_df = churn_data.corr()

    # ⑤ Save correlation matrix
    save_name = str(data_set_path).replace(
        ".csv",
        "_correlation_matrix.csv"
    )

    corr_df.to_csv(save_name)

    print(
        f"Saved correlation matrix to:\n"
        f"{save_name}"
    )

    return corr_df


# --------------------------------------------------
# Run Listing 6.2
# --------------------------------------------------

corr_df = dataset_correlation_matrix(
    data_set_path=str(data_set_path)
)


# --------------------------------------------------
# Display result
# --------------------------------------------------

display(
    corr_df.round(3)
)

Saved correlation matrix to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_correlation_matrix.csv


,account_tenure,adview_per_month,dislike_per_month,is_churn,like_per_month,message_per_month,newfriend_per_month,post_per_month,reply_per_month,unfriend_per_month
account_tenure,1.000,NaN,NaN,-0.032,0.017,NaN,NaN,NaN,NaN,NaN
adview_per_month,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dislike_per_month,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_churn,-0.032,NaN,NaN,1.000,-0.025,NaN,NaN,NaN,NaN,NaN
like_per_month,0.017,NaN,NaN,-0.025,1.000,NaN,NaN,NaN,NaN,NaN
message_per_month,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
newfriend_per_month,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
post_per_month,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
reply_per_month,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
unfriend_per_month,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# “Applying a loading matrix to a dataset in Python”

In [7]:
from pathlib import Path
from collections import Counter

import os
import numpy as np
import pandas as pd

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform


# --------------------------------------------------
# 1. Original dataset path
# --------------------------------------------------

data_set_path = (
    Path.home()
    / "Desktop"
    / "fighting-churn-learning"
    / "output"
    / "churn_dataset.csv"
)

data_set_path = str(data_set_path)


# --------------------------------------------------
# 2. Find clusters of correlated metrics
# --------------------------------------------------

def find_correlation_clusters(corr_matrix, corr_thresh=0.5):

    # Correlation 1.0 -> distance 0
    # Correlation 0.5 -> distance 0.5
    # Correlation 0.0 -> distance 1.0
    dissimilarity = 1.0 - corr_matrix

    # Numerical cleanup
    dissimilarity = dissimilarity.clip(lower=0)
    np.fill_diagonal(dissimilarity.values, 0)

    # Convert square distance matrix to condensed format
    condensed_distance = squareform(
        dissimilarity.values,
        checks=False
    )

    # Hierarchical clustering
    hierarchy = linkage(
        condensed_distance,
        method="single"
    )

    # Convert correlation threshold to distance threshold
    distance_thresh = 1.0 - corr_thresh

    labels = fcluster(
        hierarchy,
        t=distance_thresh,
        criterion="distance"
    )

    return labels


# --------------------------------------------------
# 3. Relabel groups starting from 0
# --------------------------------------------------

def relabel_clusters(labels, metric_columns):

    cluster_count = Counter(labels)

    # Largest groups first
    cluster_order = {
        cluster_label: new_label
        for new_label, (cluster_label, count)
        in enumerate(cluster_count.most_common())
    }

    relabeled = [
        cluster_order[label]
        for label in labels
    ]

    relabeled_count = Counter(relabeled)

    labeled_metrics = pd.DataFrame({
        "group": relabeled,
        "metric": metric_columns
    })

    labeled_metrics = labeled_metrics.sort_values(
        ["group", "metric"]
    ).reset_index(drop=True)

    return labeled_metrics, relabeled_count


# --------------------------------------------------
# 4. Construct loading matrix
# --------------------------------------------------

def make_load_matrix(
    labeled_metrics,
    metric_columns,
    group_counts,
    corr_thresh
):

    n_metrics = len(metric_columns)
    n_groups = len(group_counts)

    load_matrix = np.zeros(
        (n_metrics, n_groups)
    )

    for _, row in labeled_metrics.iterrows():

        metric = row["metric"]
        group = row["group"]

        metric_position = metric_columns.index(metric)
        group_size = group_counts[group]

        if group_size > 1:

            # Scale grouped metrics
            loading = 1.0 / (
                np.sqrt(corr_thresh)
                * group_size
            )

        else:

            # Single metric keeps full weight
            loading = 1.0

        load_matrix[
            metric_position,
            group
        ] = loading

    # Determine whether each output column
    # is a true multi-metric group
    group_sizes = (
        load_matrix.astype(bool)
        .sum(axis=0)
    )

    column_names = []

    for group in range(n_groups):

        if group_sizes[group] > 1:

            column_names.append(
                f"metric_group_{group + 1}"
            )

        else:

            metric_name = (
                labeled_metrics.loc[
                    labeled_metrics["group"] == group,
                    "metric"
                ]
                .iloc[0]
            )

            column_names.append(metric_name)

    load_mat_df = pd.DataFrame(
        load_matrix,
        index=metric_columns,
        columns=column_names
    )

    return load_mat_df


# --------------------------------------------------
# 5. Main Listing 6.4-style function
# --------------------------------------------------

def find_metric_groups(
    data_set_path,
    group_corr_thresh=0.5
):

    # Listing 5.3 output
    score_path = data_set_path.replace(
        ".csv",
        "_scores.csv"
    )

    assert os.path.isfile(score_path), \
        f"Score file not found:\n{score_path}"

    # Load standardized scores
    score_data = pd.read_csv(
        score_path,
        index_col=[0, 1]
    )

    # Remove target
    if "is_churn" in score_data.columns:
        score_data = score_data.drop(
            columns=["is_churn"]
        )

    # Keep numeric metrics
    score_data = score_data.select_dtypes(
        include="number"
    )

    metric_columns = score_data.columns.tolist()

    print("Metrics being grouped:")
    for metric in metric_columns:
        print(" -", metric)

    # Correlation matrix
    corr_matrix = score_data.corr()

    # Find correlated clusters
    labels = find_correlation_clusters(
        corr_matrix,
        corr_thresh=group_corr_thresh
    )

    # Relabel groups
    labeled_metrics, group_counts = (
        relabel_clusters(
            labels,
            metric_columns
        )
    )

    # Build load matrix
    load_mat_df = make_load_matrix(
        labeled_metrics,
        metric_columns,
        group_counts,
        group_corr_thresh
    )

    # --------------------------------------------------
    # Save loading matrix
    # --------------------------------------------------

    load_mat_path = data_set_path.replace(
        ".csv",
        "_load_mat.csv"
    )

    load_mat_df.to_csv(
        load_mat_path
    )

    # --------------------------------------------------
    # Save human-readable group definitions
    # --------------------------------------------------

    group_rows = []

    for group_id in sorted(
        labeled_metrics["group"].unique()
    ):

        metrics = (
            labeled_metrics.loc[
                labeled_metrics["group"] == group_id,
                "metric"
            ]
            .tolist()
        )

        output_name = load_mat_df.columns[group_id]

        group_rows.append({
            "group": output_name,
            "metrics": " | ".join(metrics)
        })

    group_df = pd.DataFrame(group_rows)

    group_path = data_set_path.replace(
        ".csv",
        "_groupmets.csv"
    )

    group_df.to_csv(
        group_path,
        index=False
    )

    print(
        f"\nLoading matrix saved to:\n"
        f"{load_mat_path}"
    )

    print(
        f"\nMetric groups saved to:\n"
        f"{group_path}"
    )

    return load_mat_df, group_df


# --------------------------------------------------
# 6. Run Listing 6.4
# --------------------------------------------------

load_mat_df, metric_groups = find_metric_groups(
    data_set_path=data_set_path,
    group_corr_thresh=0.5
)


# --------------------------------------------------
# 7. Display results
# --------------------------------------------------

print("\nLoading matrix:")
display(load_mat_df)

print("\nMetric groups:")
display(metric_groups)

Metrics being grouped:
 - like_per_month
 - account_tenure

Loading matrix saved to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_load_mat.csv

Metric groups saved to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_groupmets.csv

Loading matrix:


,like_per_month,account_tenure
like_per_month,1.0,0.0
account_tenure,0.0,1.0



Metric groups:


,group,metrics
0,like_per_month,like_per_month
1,account_tenure,account_tenure


In [8]:
# --------------------------------------------------
# Define function
# --------------------------------------------------

def apply_metric_groups(data_set_path):

    # ① Score file created by Listing 5.3
    score_save_path = str(data_set_path).replace(
        ".csv",
        "_scores.csv"
    )

    assert os.path.isfile(score_save_path), \
        "Run Listing 5.3 first to generate metric scores."

    # ② Load standardized metric scores
    score_data = pd.read_csv(
        score_save_path,
        index_col=[0, 1]
    )

    # ③ Temporarily remove churn target
    if "is_churn" not in score_data.columns:
        raise ValueError(
            '"is_churn" column not found in score dataset.'
        )

    data_2group = score_data.drop(
        columns=["is_churn"]
    )

    # ④ Load loading matrix created by Listing 6.4
    load_mat_path = str(data_set_path).replace(
        ".csv",
        "_load_mat.csv"
    )

    assert os.path.isfile(load_mat_path), \
        "Run Listing 6.4 first to generate the loading matrix."

    load_mat_df = pd.read_csv(
        load_mat_path,
        index_col=0
    )

    # --------------------------------------------------
    # Validate feature compatibility
    # --------------------------------------------------

    missing_features = [
        feature
        for feature in load_mat_df.index
        if feature not in data_2group.columns
    ]

    if missing_features:
        raise ValueError(
            "The following loading-matrix features "
            "are missing from the score dataset:\n"
            + "\n".join(missing_features)
        )

    # ⑤ Convert loading matrix to NumPy array
    load_mat_ndarray = load_mat_df.to_numpy(
        dtype=float
    )

    # ⑥ Reorder metric columns to exactly match
    # loading-matrix row order
    ndarray_2group = (
        data_2group[
            load_mat_df.index
        ]
        .to_numpy(dtype=float)
    )

    # --------------------------------------------------
    # Shape check
    # --------------------------------------------------

    print("Metric-score matrix shape:")
    print(ndarray_2group.shape)

    print("\nLoading matrix shape:")
    print(load_mat_ndarray.shape)

    # ⑦ Matrix multiplication
    grouped_ndarray = np.matmul(
        ndarray_2group,
        load_mat_ndarray
    )

    # ⑧ Convert grouped array back to DataFrame
    churn_data_grouped = pd.DataFrame(
        grouped_ndarray,
        columns=load_mat_df.columns,
        index=score_data.index
    )

    # ⑨ Add churn target back
    churn_data_grouped["is_churn"] = (
        score_data["is_churn"]
    )

    # ⑩ Save grouped dataset
    save_path = str(data_set_path).replace(
        ".csv",
        "_groupscore.csv"
    )

    churn_data_grouped.to_csv(
        save_path,
        header=True
    )

    print(
        f"\nSaved grouped data to:\n{save_path}"
    )

    return churn_data_grouped


# --------------------------------------------------
# 3. Run function
# --------------------------------------------------

grouped_data = apply_metric_groups(
    data_set_path=str(data_set_path)
)


# --------------------------------------------------
# 4. Inspect result
# --------------------------------------------------

print("\nShape:")
print(grouped_data.shape)

print("\nColumns:")
print(grouped_data.columns.tolist())

display(grouped_data.head())

Metric-score matrix shape:
(32897, 2)

Loading matrix shape:
(2, 2)

Saved grouped data to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_groupscore.csv

Shape:
(32897, 3)

Columns:
['like_per_month', 'account_tenure', 'is_churn']


,,like_per_month,account_tenure,is_churn
,account_id,,,
0,-1.680473,0.516299,-1.235197,0.0
1,-1.668907,-1.429966,-1.235197,0.0
2,-1.650979,-1.135170,-1.235197,0.0
3,-1.628425,-0.677778,-1.235197,0.0
4,-1.620040,-0.830943,-1.235197,0.0


In [9]:
from pathlib import Path

output_dir = (
    Path.home()
    / "Desktop"
    / "fighting-churn-learning"
    / "output"
)

for file in sorted(output_dir.glob("*")):
    print(file.name)

.DS_Store
churn_dataset.csv
churn_dataset_backtest.csv
churn_dataset_chapter7.csv
churn_dataset_cleaned.csv
churn_dataset_correlation_matrix.csv
churn_dataset_crossval.csv
churn_dataset_crossval_folds.csv
churn_dataset_crossval_regression_auc.png
churn_dataset_crossval_regression_lift.png
churn_dataset_crossval_regression_weights.png
churn_dataset_crossval_xgb.csv
churn_dataset_current.csv
churn_dataset_current_churnhist.csv
churn_dataset_current_clipping_summary.csv
churn_dataset_current_groupmets_segment.csv
churn_dataset_current_groupscore.csv
churn_dataset_current_predictions.csv
churn_dataset_current_scores.csv
churn_dataset_current_xgb_churnhist.csv
churn_dataset_current_xgb_predictions.csv
churn_dataset_demographic.csv
churn_dataset_demographic_channel_churn_category.csv
churn_dataset_demographic_channel_churn_category.png
churn_dataset_demographic_country_group_churn_category.csv
churn_dataset_demographic_country_group_churn_category.png
churn_dataset_demographic_current.csv
ch

# Calculating the correlation matrix for a dataset in Python

In [10]:
def dataset_correlation_matrix(data_set_path):

    # ① Check path
    assert os.path.isfile(data_set_path), \
        f'"{data_set_path}" is not a valid path'

    # ② Load dataset
    churn_data = pd.read_csv(
        data_set_path,
        index_col=[0, 1]
    )

    # Convert churn target to numeric if needed
    if "is_churn" in churn_data.columns:
        if churn_data["is_churn"].dtype == object:
            churn_data["is_churn"] = (
                churn_data["is_churn"]
                .astype(str)
                .str.lower()
                .map({
                    "true": 1.0,
                    "false": 0.0,
                    "t": 1.0,
                    "f": 0.0,
                    "1": 1.0,
                    "0": 0.0
                })
            )
        else:
            churn_data["is_churn"] = (
                churn_data["is_churn"]
                .astype(float)
            )

    # Keep numeric columns only
    churn_data = churn_data.select_dtypes(
        include="number"
    )

    # ③ Sort columns alphabetically
    churn_data = churn_data.reindex(
        sorted(churn_data.columns),
        axis=1
    )

    # ④ Calculate Pearson correlation matrix
    corr_df = churn_data.corr()

    # ⑤ Save result
    save_name = str(data_set_path).replace(
        ".csv",
        "_correlation_matrix.csv"
    )

    corr_df.to_csv(save_name)

    print(
        f"Saved correlation matrix to:\n"
        f"{save_name}"
    )

    return corr_df


corr_df = dataset_correlation_matrix(
    data_set_path=str(data_set_path)
)

display(corr_df.round(3))

Saved correlation matrix to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_correlation_matrix.csv


,account_tenure,adview_per_month,dislike_per_month,is_churn,like_per_month,message_per_month,newfriend_per_month,post_per_month,reply_per_month,unfriend_per_month
account_tenure,1.000,NaN,NaN,-0.032,0.017,NaN,NaN,NaN,NaN,NaN
adview_per_month,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dislike_per_month,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_churn,-0.032,NaN,NaN,1.000,-0.025,NaN,NaN,NaN,NaN,NaN
like_per_month,0.017,NaN,NaN,-0.025,1.000,NaN,NaN,NaN,NaN,NaN
message_per_month,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
newfriend_per_month,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
post_per_month,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
reply_per_month,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
unfriend_per_month,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Applying a loading matrix to a dataset in Python


In [11]:

# --------------------------------------------------
# 2. Define Listing 6.3
# --------------------------------------------------

def apply_metric_groups(data_set_path):

    # ① Find score file produced by Listing 5.3
    score_save_path = data_set_path.replace(
        ".csv",
        "_scores.csv"
    )

    assert os.path.isfile(score_save_path), \
        f"Run Listing 5.3 first. Missing:\n{score_save_path}"

    # ② Load standardized metric scores
    score_data = pd.read_csv(
        score_save_path,
        index_col=[0, 1]
    )

    # Check target exists
    if "is_churn" not in score_data.columns:
        raise ValueError(
            '"is_churn" was not found in the score dataset.'
        )

    # ③ Remove target temporarily
    data_2group = score_data.drop(
        columns=["is_churn"]
    )

    # --------------------------------------------------
    # ④ Load matrix produced by Listing 6.4
    # --------------------------------------------------

    load_mat_path = data_set_path.replace(
        ".csv",
        "_load_mat.csv"
    )

    assert os.path.isfile(load_mat_path), \
        f"Run Listing 6.4 first. Missing:\n{load_mat_path}"

    load_mat_df = pd.read_csv(
        load_mat_path,
        index_col=0
    )

    print("Score features:")
    print(data_2group.columns.tolist())

    print("\nLoading matrix features:")
    print(load_mat_df.index.tolist())

    # --------------------------------------------------
    # Check loading-matrix features exist
    # --------------------------------------------------

    missing_metrics = [
        metric
        for metric in load_mat_df.index
        if metric not in data_2group.columns
    ]

    if missing_metrics:
        raise ValueError(
            "Loading matrix contains metrics that are "
            "missing from the score dataset:\n"
            + "\n".join(missing_metrics)
        )

    # --------------------------------------------------
    # ⑤ Convert loading matrix to NumPy
    # --------------------------------------------------

    load_mat_ndarray = load_mat_df.to_numpy(
        dtype=float
    )

    # --------------------------------------------------
    # ⑥ Put score columns into exactly the same order
    # as the loading-matrix rows
    # --------------------------------------------------

    ndarray_2group = (
        data_2group[
            load_mat_df.index
        ]
        .to_numpy(dtype=float)
    )

    print("\nScore matrix shape:")
    print(ndarray_2group.shape)

    print("\nLoading matrix shape:")
    print(load_mat_ndarray.shape)

    # --------------------------------------------------
    # ⑦ Apply the metric grouping
    # --------------------------------------------------

    grouped_ndarray = np.matmul(
        ndarray_2group,
        load_mat_ndarray
    )

    print("\nGrouped matrix shape:")
    print(grouped_ndarray.shape)

    # --------------------------------------------------
    # ⑧ Convert NumPy result back to DataFrame
    # --------------------------------------------------

    churn_data_grouped = pd.DataFrame(
        grouped_ndarray,
        columns=load_mat_df.columns,
        index=score_data.index
    )

    # --------------------------------------------------
    # ⑨ Reattach churn target
    # --------------------------------------------------

    churn_data_grouped["is_churn"] = (
        score_data["is_churn"]
    )

    # --------------------------------------------------
    # ⑩ Save result
    # --------------------------------------------------

    save_path = data_set_path.replace(
        ".csv",
        "_groupscore.csv"
    )

    churn_data_grouped.to_csv(
        save_path,
        header=True
    )

    print(
        f"\nSaved grouped data to:\n{save_path}"
    )

    return churn_data_grouped


# --------------------------------------------------
# 3. Run Listing 6.3
# --------------------------------------------------

grouped_data = apply_metric_groups(
    data_set_path=data_set_path
)


# --------------------------------------------------
# 4. Inspect result
# --------------------------------------------------

print("\nFinal dataset shape:")
print(grouped_data.shape)

print("\nFinal columns:")
print(grouped_data.columns.tolist())

display(grouped_data.head())

Score features:
['like_per_month', 'account_tenure']

Loading matrix features:
['like_per_month', 'account_tenure']

Score matrix shape:
(32897, 2)

Loading matrix shape:
(2, 2)

Grouped matrix shape:
(32897, 2)

Saved grouped data to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_groupscore.csv

Final dataset shape:
(32897, 3)

Final columns:
['like_per_month', 'account_tenure', 'is_churn']


,,like_per_month,account_tenure,is_churn
,account_id,,,
0,-1.680473,0.516299,-1.235197,0.0
1,-1.668907,-1.429966,-1.235197,0.0
2,-1.650979,-1.135170,-1.235197,0.0
3,-1.628425,-0.677778,-1.235197,0.0
4,-1.620040,-0.830943,-1.235197,0.0


# Finding metric groups and creating a loading matrix in Python

In [12]:
# --------------------------------------------------
# Find correlation-based clusters
# --------------------------------------------------

def find_correlation_clusters(corr, corr_thresh):

    # ② Convert similarity (correlation)
    # into dissimilarity
    dissimilarity = 1.0 - corr

    # Protect against tiny floating-point errors
    dissimilarity = dissimilarity.clip(lower=0)

    # Diagonal must be exactly zero for distance matrix
    np.fill_diagonal(
        dissimilarity.values,
        0.0
    )

    # ③ Convert correlation threshold
    # into distance threshold
    diss_thresh = 1.0 - corr_thresh

    # ④ Hierarchical clustering
    hierarchy = linkage(
        squareform(
            dissimilarity.values,
            checks=False
        ),
        method="single"
    )

    # ⑤ Cut hierarchy into clusters
    labels = fcluster(
        hierarchy,
        diss_thresh,
        criterion="distance"
    )

    return labels


# --------------------------------------------------
# Relabel clusters
# --------------------------------------------------

def relabel_clusters(
    labels,
    metric_columns
):

    # ⑥ Count metrics in each cluster
    cluster_count = Counter(labels)

    # ⑦ Put largest clusters first
    cluster_order = {
        cluster[0]: idx
        for idx, cluster
        in enumerate(
            cluster_count.most_common()
        )
    }

    # ⑧ Replace original labels
    # with ordered labels 0,1,2,...
    relabeled_clusters = [
        cluster_order[label]
        for label in labels
    ]

    # ⑨ Count relabeled clusters
    relabeled_count = Counter(
        relabeled_clusters
    )

    # ⑩ DataFrame showing which metric
    # belongs to which group
    labeled_column_df = pd.DataFrame({
        "group": relabeled_clusters,
        "column": metric_columns
    })

    labeled_column_df = (
        labeled_column_df
        .sort_values(
            ["group", "column"],
            ascending=[True, True]
        )
        .reset_index(drop=True)
    )

    return (
        labeled_column_df,
        relabeled_count
    )


# --------------------------------------------------
# Create loading matrix
# --------------------------------------------------

def make_load_matrix(
    labeled_column_df,
    metric_columns,
    relabeled_count,
    corr_thresh
):

    # ⑪ Matrix:
    # rows = original metrics
    # columns = metric groups
    load_mat = np.zeros(
        (
            len(metric_columns),
            len(relabeled_count)
        )
    )

    # ⑫ Fill loading weights
    for _, row in labeled_column_df.iterrows():

        group_id = int(row["group"])
        metric_name = row["column"]

        orig_col = metric_columns.index(
            metric_name
        )

        # ⑬ Is this a multi-metric group?
        if relabeled_count[group_id] > 1:

            # ⑭ Group weighting
            load_mat[
                orig_col,
                group_id
            ] = (
                1.0
                /
                (
                    np.sqrt(corr_thresh)
                    * float(
                        relabeled_count[group_id]
                    )
                )
            )

        else:

            # ⑮ Single feature keeps weight 1
            load_mat[
                orig_col,
                group_id
            ] = 1.0

    # ⑯ Which output columns
    # contain multiple metrics?
    is_group = (
        load_mat
        .astype(bool)
        .sum(axis=0)
        > 1
    )

    # ⑰ / ⑱ Create column names
    column_names = []

    for group_id in range(
        load_mat.shape[1]
    ):

        if is_group[group_id]:

            column_names.append(
                f"metric_group_{group_id + 1}"
            )

        else:

            metric_name = (
                labeled_column_df.loc[
                    labeled_column_df["group"]
                    == group_id,
                    "column"
                ]
                .iloc[0]
            )

            column_names.append(
                metric_name
            )

    # ⑲ Convert matrix to DataFrame
    loadmat_df = pd.DataFrame(
        load_mat,
        index=metric_columns,
        columns=column_names
    )

    # ⑳ Temporary name column
    # used only for sorting
    loadmat_df["name"] = (
        loadmat_df.index
    )

    # ㉑ Sort columns
    sort_cols = list(
        loadmat_df.columns
    )

    # ㉒ Descending for loading columns
    sort_order = (
        [False]
        * loadmat_df.shape[1]
    )

    # ㉓ Metric name ascending
    sort_order[-1] = True

    # ㉔ Sort for readability
    loadmat_df = (
        loadmat_df
        .sort_values(
            sort_cols,
            ascending=sort_order
        )
    )

    # ㉕ Remove temporary name column
    loadmat_df = loadmat_df.drop(
        columns=["name"]
    )

    return loadmat_df


# --------------------------------------------------
# Main Listing 6.4 function
# --------------------------------------------------

def find_metric_groups(
    data_set_path,
    group_corr_thresh=0.5
):

    # ㉖ Load standardized scores
    score_save_path = (
        data_set_path.replace(
            ".csv",
            "_scores.csv"
        )
    )

    assert os.path.isfile(
        score_save_path
    ), (
        "You must run Listing 5.3 "
        "to create metric scores first."
    )

    score_data = pd.read_csv(
        score_save_path,
        index_col=[0, 1]
    )

    # Remove churn target
    if "is_churn" in score_data.columns:

        score_data = score_data.drop(
            columns=["is_churn"]
        )

    # Keep only numeric metrics
    score_data = (
        score_data
        .select_dtypes(
            include="number"
        )
    )

    # ㉗ Original feature names
    metric_columns = list(
        score_data.columns
    )

    print("Metrics:")
    for metric in metric_columns:
        print(f" - {metric}")

    # ㉘ Find correlation groups
    labels = find_correlation_clusters(
        score_data.corr(),
        group_corr_thresh
    )

    (
        labeled_column_df,
        relabeled_count
    ) = relabel_clusters(
        labels,
        metric_columns
    )

    print("\nMetric group assignments:")
    display(labeled_column_df)

    # Build loading matrix
    loadmat_df = make_load_matrix(
        labeled_column_df,
        metric_columns,
        relabeled_count,
        group_corr_thresh
    )

    # Save loading matrix
    load_mat_path = (
        data_set_path.replace(
            ".csv",
            "_load_mat.csv"
        )
    )

    print(
        f"\nSaving loadings to:\n"
        f"{load_mat_path}"
    )

    loadmat_df.to_csv(
        load_mat_path
    )

    # ㉙ Human-readable metric groups
    group_lists = []

    group_ids = sorted(
        labeled_column_df[
            "group"
        ].unique()
    )

    for group_id in group_ids:

        metrics = (
            labeled_column_df.loc[
                labeled_column_df["group"]
                == group_id,
                "column"
            ]
            .tolist()
        )

        group_lists.append(
            "|".join(metrics)
        )

    # Important:
    # loading-matrix columns correspond
    # to the group order
    groupmet_df = pd.DataFrame(
        group_lists,
        index=loadmat_df.columns,
        columns=["metrics"]
    )

    # ㉚ Save metric-group definitions
    groupmet_path = (
        data_set_path.replace(
            ".csv",
            "_groupmets.csv"
        )
    )

    print(
        f"\nSaving metric groups to:\n"
        f"{groupmet_path}"
    )

    groupmet_df.to_csv(
        groupmet_path
    )

    return (
        loadmat_df,
        groupmet_df
    )


# --------------------------------------------------
# Run Listing 6.4
# --------------------------------------------------

loadmat_df, groupmet_df = (
    find_metric_groups(
        data_set_path=data_set_path,
        group_corr_thresh=0.5
    )
)


# --------------------------------------------------
# Inspect results
# --------------------------------------------------

print("\nLoading matrix:")
display(loadmat_df)

print("\nMetric groups:")
display(groupmet_df)

Metrics:
 - like_per_month
 - account_tenure

Metric group assignments:


,group,column
0,0,like_per_month
1,1,account_tenure



Saving loadings to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_load_mat.csv

Saving metric groups to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_groupmets.csv

Loading matrix:


,like_per_month,account_tenure
like_per_month,1.0,0.0
account_tenure,0.0,1.0



Metric groups:


,metrics
like_per_month,like_per_month
account_tenure,account_tenure


# Creating an ordered correlation matrix

In [13]:
# --------------------------------------------------
# Define ordered correlation matrix function
# --------------------------------------------------

def ordered_correlation_matrix(data_set_path):

    # --------------------------------------------------
    # ① Load standardized scores from Listing 5.3
    # --------------------------------------------------

    score_path = data_set_path.replace(
        ".csv",
        "_scores.csv"
    )

    assert os.path.isfile(score_path), \
        f"Score file not found:\n{score_path}"

    churn_data = pd.read_csv(
        score_path,
        index_col=[0, 1]
    )

    # --------------------------------------------------
    # Load loading matrix from Listing 6.4
    # --------------------------------------------------

    load_mat_path = data_set_path.replace(
        ".csv",
        "_load_mat.csv"
    )

    assert os.path.isfile(load_mat_path), \
        f"Loading matrix not found:\n{load_mat_path}"

    load_mat_df = pd.read_csv(
        load_mat_path,
        index_col=0
    )

    # --------------------------------------------------
    # ② Reorder metrics according to loading matrix
    # --------------------------------------------------

    metric_order = (
        load_mat_df.index.tolist()
    )

    # Make sure all loading-matrix metrics
    # exist in score dataset
    missing_metrics = [
        metric
        for metric in metric_order
        if metric not in churn_data.columns
    ]

    if missing_metrics:
        raise ValueError(
            "The following loading-matrix metrics "
            "are missing from the score dataset:\n"
            + "\n".join(missing_metrics)
        )

    churn_data = churn_data[
        metric_order
    ]

    # --------------------------------------------------
    # Calculate correlation matrix
    # --------------------------------------------------

    corr = churn_data.corr()

    # --------------------------------------------------
    # Save result
    # --------------------------------------------------

    save_name = data_set_path.replace(
        ".csv",
        "_ordered_correlation_matrix.csv"
    )

    corr.to_csv(
        save_name
    )

    print(
        f"Saved correlation matrix to:\n"
        f"{save_name}"
    )

    return corr


# --------------------------------------------------
# Run function
# --------------------------------------------------

ordered_corr = ordered_correlation_matrix(
    data_set_path=data_set_path
)


# --------------------------------------------------
# Display result
# --------------------------------------------------

display(
    ordered_corr.round(3)
)

Saved correlation matrix to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_ordered_correlation_matrix.csv


,like_per_month,account_tenure
like_per_month,1.000,0.021
account_tenure,0.021,1.000
